In [8]:
# ============================================================
#  Baseline N-gram Language Model
#  Supports: 3-gram, 5-gram, perplexity, coverage, rare words
# ============================================================

import re
import math
import random
from collections import Counter, defaultdict
from typing import List, Tuple, Dict, Optional
import nltk
from nltk.util import ngrams
nltk.download('punkt', quiet=True)

# ─────────────────────────────────────────────
#  1. TEXT PREPROCESSING
# ─────────────────────────────────────────────

RARE_THRESHOLD = 3          # слова встречающиеся < N раз → <UNK>
UNK_TOKEN      = "<UNK>"
BOS_TOKEN      = "<BOS>"    # begin of sentence
EOS_TOKEN      = "<EOS>"    # end of sentence

def tokenize(text: str) -> List[str]:
    """Простая токенизация: нижний регистр + слова/пунктуация."""
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens

def build_vocab(corpus: List[List[str]],
                rare_threshold: int = RARE_THRESHOLD) -> Dict[str, int]:
    """
    Строит словарь.
    Слова с частотой < rare_threshold заменяются на <UNK>.
    Возвращает word2freq.
    """
    freq = Counter(token for sent in corpus for token in sent)
    vocab = {word: cnt for word, cnt in freq.items()
             if cnt >= rare_threshold}
    vocab[UNK_TOKEN] = sum(cnt for word, cnt in freq.items()
                           if cnt < rare_threshold)
    return vocab

def replace_rare(corpus: List[List[str]],
                 vocab: Dict[str, int]) -> List[List[str]]:
    """Заменяет OOV / редкие слова на <UNK>."""
    return [
        [token if token in vocab else UNK_TOKEN for token in sent]
        for sent in corpus
    ]

def add_padding(corpus: List[List[str]], n: int) -> List[List[str]]:
    """Добавляет (n-1) BOS и 1 EOS к каждому предложению."""
    pad = [BOS_TOKEN] * (n - 1)
    return [pad + sent + [EOS_TOKEN] for sent in corpus]


# ─────────────────────────────────────────────
#  2. N-GRAM MODEL
# ─────────────────────────────────────────────

class NGramModel:
    """
    Baseline N-gram Language Model с Add-k (Laplace) сглаживанием.
    """

    def __init__(self, n: int = 3, k: float = 1.0):
        assert n >= 2, "n должно быть >= 2"
        self.n       = n          # порядок модели
        self.k       = k          # коэффициент сглаживания
        self.ngram_counts:   Counter = Counter()
        self.context_counts: Counter = Counter()
        self.vocab:   Dict[str, int] = {}
        self.vocab_size: int = 0

    # ---------- обучение ----------

    def fit(self, corpus: List[List[str]], vocab: Dict[str, int]):
        """
        corpus  — список предложений (уже с <UNK> и паддингом)
        vocab   — словарь {word: freq}
        """
        self.vocab      = vocab
        self.vocab_size = len(vocab)

        for sentence in corpus:
            for ngram in ngrams(sentence, self.n):
                self.ngram_counts[ngram]       += 1
                self.context_counts[ngram[:-1]] += 1

        print(f"[NGramModel n={self.n}] обучен: "
              f"{len(self.ngram_counts):,} {self.n}-грамм, "
              f"словарь={self.vocab_size:,} слов")

    # ---------- вероятность ----------

    def prob(self, word: str, context: Tuple[str, ...]) -> float:
        """
        P(word | context) с Add-k сглаживанием.
        context — кортеж из (n-1) предшествующих токенов.
        """
        # OOV → <UNK>
        word    = word    if word    in self.vocab else UNK_TOKEN
        context = tuple(w if w in self.vocab else UNK_TOKEN
                        for w in context)

        ngram_cnt   = self.ngram_counts.get(context + (word,), 0)
        context_cnt = self.context_counts.get(context, 0)

        return (ngram_cnt + self.k) / \
               (context_cnt + self.k * self.vocab_size)

    def log_prob(self, word: str, context: Tuple[str, ...]) -> float:
        p = self.prob(word, context)
        return math.log2(p) if p > 0 else float('-inf')

    # ---------- sentence log-probability ----------

    def sentence_log_prob(self, sentence: List[str]) -> float:
        """Log₂ P(sentence) для уже дополненного предложения."""
        total = 0.0
        for i in range(self.n - 1, len(sentence)):
            word    = sentence[i]
            context = tuple(sentence[i - (self.n - 1): i])
            total  += self.log_prob(word, context)
        return total


# ─────────────────────────────────────────────
#  3. PERPLEXITY
# ─────────────────────────────────────────────

def compute_perplexity(model: NGramModel,
                       test_corpus: List[List[str]]) -> float:
    """
    PP(W) = 2^( -1/N * Σ log₂ P(wᵢ|context) )
    N — общее число токенов в тесте (без BOS-паддинга).
    """
    total_log_prob = 0.0
    total_tokens   = 0

    for sentence in test_corpus:
        # считаем токены без BOS-паддинга
        real_tokens = len(sentence) - (model.n - 1)
        if real_tokens <= 0:
            continue
        total_log_prob += model.sentence_log_prob(sentence)
        total_tokens   += real_tokens

    if total_tokens == 0:
        return float('inf')

    avg_log_prob = total_log_prob / total_tokens
    perplexity   = 2 ** (-avg_log_prob)
    return perplexity


# ─────────────────────────────────────────────
#  4. COVERAGE
# ─────────────────────────────────────────────

def compute_coverage(model: NGramModel,
                     test_corpus: List[List[str]]) -> Dict:
    """
    Считает:
      - token_coverage  : % токенов теста, известных модели
      - type_coverage   : % уникальных слов теста, известных модели
      - oov_rate        : % токенов, попавших в <UNK>
      - ngram_coverage  : % n-грамм теста, встреченных при обучении
    """
    test_tokens = [tok for sent in test_corpus
                   for tok in sent
                   if tok not in (BOS_TOKEN, EOS_TOKEN)]

    known_tokens = sum(1 for t in test_tokens if t in model.vocab)
    oov_tokens   = len(test_tokens) - known_tokens

    test_types   = set(test_tokens)
    known_types  = test_types & set(model.vocab.keys())

    # n-gram coverage
    test_ngrams  = [ng for sent in test_corpus
                    for ng in ngrams(sent, model.n)]
    known_ngrams = sum(1 for ng in test_ngrams
                       if ng in model.ngram_counts)

    return {
        "token_coverage_%"  : round(100 * known_tokens / max(len(test_tokens), 1), 2),
        "type_coverage_%"   : round(100 * len(known_types) / max(len(test_types), 1), 2),
        "oov_rate_%"        : round(100 * oov_tokens / max(len(test_tokens), 1), 2),
        "ngram_coverage_%"  : round(100 * known_ngrams / max(len(test_ngrams), 1), 2),
        "total_test_tokens" : len(test_tokens),
        "oov_tokens"        : oov_tokens,
    }


# ─────────────────────────────────────────────
#  5. RARE WORD ANALYSIS
# ─────────────────────────────────────────────

def rare_word_analysis(raw_corpus: List[List[str]],
                       vocab: Dict[str, int],
                       rare_threshold: int = RARE_THRESHOLD) -> Dict:
    """
    Статистика по редким словам в корпусе.
    """
    all_tokens = [tok for sent in raw_corpus for tok in sent]
    freq       = Counter(all_tokens)

    rare_words  = {w for w, c in freq.items() if c < rare_threshold}
    total_types = len(freq)
    total_tok   = len(all_tokens)
    rare_tokens = sum(freq[w] for w in rare_words)

    # топ-10 редких слов
    top_rare = sorted(rare_words, key=lambda w: freq[w], reverse=True)[:10]

    return {
        "total_types"        : total_types,
        "rare_types"         : len(rare_words),
        "rare_type_%"        : round(100 * len(rare_words) / max(total_types, 1), 2),
        "rare_token_count"   : rare_tokens,
        "rare_token_%"       : round(100 * rare_tokens / max(total_tok, 1), 2),
        "unk_freq_in_vocab"  : vocab.get(UNK_TOKEN, 0),
        "top_rare_words"     : top_rare,
        "rare_threshold"     : rare_threshold,
    }


# ─────────────────────────────────────────────
#  6. FULL PIPELINE
# ─────────────────────────────────────────────

def run_pipeline(raw_text: str,
                 test_size: float = 0.2,
                 rare_threshold: int = RARE_THRESHOLD,
                 smoothing_k: float = 1.0,
                 seed: int = 42):

    print("=" * 60)
    print("  N-GRAM BASELINE PIPELINE")
    print("=" * 60)

    # ── 1. Токенизация по предложениям ─────────────────────────
    sentences_raw = [tokenize(s)
                 for line in raw_text.splitlines()      # сначала по строкам
                 for s in re.split(r'[.!?]', line)      # потом по .!?
                 if len(tokenize(s)) > 1]

    random.seed(seed)
    random.shuffle(sentences_raw)

    split = int(len(sentences_raw) * (1 - test_size))
    train_raw = sentences_raw[:split]
    test_raw  = sentences_raw[split:]

    print(f"\n📄 Корпус: {len(sentences_raw)} предложений  "
          f"(train={len(train_raw)}, test={len(test_raw)})")

    # ── 2. Словарь и <UNK> ─────────────────────────────────────
    vocab = build_vocab(train_raw, rare_threshold)

    rare_stats = rare_word_analysis(train_raw, vocab, rare_threshold)
    print(f"\n{'─'*50}")
    print("📊 RARE WORD ANALYSIS (train)")
    for k, v in rare_stats.items():
        print(f"   {k:<25}: {v}")

    # Применяем <UNK> к обоим корпусам на основе ТРЕНИРОВОЧНОГО словаря
    train_unk = replace_rare(train_raw, vocab)
    test_unk  = replace_rare(test_raw,  vocab)   # OOV из теста → <UNK>

    # ── 3. Обучаем и оцениваем 3-gram и 5-gram ─────────────────
    results = {}

    for n in [3, 5]:
        print(f"\n{'─'*50}")
        print(f"🔧 N={n}-GRAM MODEL  (Add-{smoothing_k} smoothing)")

        train_pad = add_padding(train_unk, n)
        test_pad  = add_padding(test_unk,  n)

        model = NGramModel(n=n, k=smoothing_k)
        model.fit(train_pad, vocab)

        # Perplexity
        pp = compute_perplexity(model, test_pad)
        print(f"\n   📉 Perplexity (test)        : {pp:.4f}")

        # Coverage
        cov = compute_coverage(model, test_pad)
        print(f"\n   🌐 COVERAGE (test)")
        for ck, cv in cov.items():
            print(f"      {ck:<28}: {cv}")

        results[f"{n}-gram"] = {"perplexity": pp, "coverage": cov}

    # ── 4. Сравнительная таблица ────────────────────────────────
    print(f"\n{'='*60}")
    print("📋 COMPARISON TABLE")
    print(f"{'Model':<12} {'Perplexity':>14} {'Token Cov%':>12} {'OOV%':>8} {'N-gram Cov%':>14}")
    print("-" * 60)
    for model_name, res in results.items():
        pp  = res["perplexity"]
        cov = res["coverage"]
        print(f"{model_name:<12} {pp:>14.4f} "
              f"{cov['token_coverage_%']:>12} "
              f"{cov['oov_rate_%']:>8} "
              f"{cov['ngram_coverage_%']:>14}")

    return results


# ─────────────────────────────────────────────
#  7. DEMO
# ─────────────────────────────────────────────

if __name__ == "__main__":
    with open("gaming_corpus_clean.txt", "r", encoding="utf-8") as f:
        raw_text = f.read() 
    run_pipeline(
        raw_text       = raw_text,
        test_size      = 0.2,
        rare_threshold = 2,     # слова < 2 раза → <UNK>
        smoothing_k    = 1.0,   # Laplace smoothing
    )


  N-GRAM BASELINE PIPELINE

📄 Корпус: 7609 предложений  (train=6087, test=1522)

──────────────────────────────────────────────────
📊 RARE WORD ANALYSIS (train)
   total_types              : 9022
   rare_types               : 4864
   rare_type_%              : 53.91
   rare_token_count         : 4864
   rare_token_%             : 4.44
   unk_freq_in_vocab        : 4864
   top_rare_words           : ['laying', 'fabric', 'viet', 'slick', 'ridiculed', 'profishency', 'thumbsup', 'tuyet', 'alwayswannafly', 'wowww']
   rare_threshold           : 2

──────────────────────────────────────────────────
🔧 N=3-GRAM MODEL  (Add-1.0 smoothing)
[NGramModel n=3] обучен: 82,861 3-грамм, словарь=4,159 слов

   📉 Perplexity (test)        : 2294.6036

   🌐 COVERAGE (test)
      token_coverage_%            : 100.0
      type_coverage_%             : 100.0
      oov_rate_%                  : 0.0
      ngram_coverage_%            : 38.06
      total_test_tokens           : 29805
      oov_tokens             